In [33]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # Use GPU if available
device

device(type='cuda', index=0)

In [35]:
words = open('names.txt', 'r').read().splitlines()
print(f"First eight names in dataset:\t\t{words[:8]}")
print(f"Count of names in dataset:\t\t{len(words)}")
print(f"Count of unique characters in dataset:\t{len(list(set(''.join(words))))}")

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

# Showing the first 10 entries of the two mappings
# (they just mirror each other)
print(f"Integer-to-Character Map: {list(itos.items())[:10]}")
print(f"Character-to-Integer Map: {list(stoi.items())[:10]}")

First eight names in dataset:		['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
Count of names in dataset:		32033
Count of unique characters in dataset:	26
Integer-to-Character Map: [(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd'), (5, 'e'), (6, 'f'), (7, 'g'), (8, 'h'), (9, 'i'), (10, 'j')]
Character-to-Integer Map: [('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5), ('f', 6), ('g', 7), ('h', 8), ('i', 9), ('j', 10)]


In [37]:
# Building the dataset
block_size = 3 # context length: how many characters are used to predict the next? (this was 1 before)
X, Y = [], []  # features (input, the context) and labels (output, the next character)

# Iterating over the first five names
# Splitting each into characters getting mapped to their indices
# and then forming the input context and the expected output character entry for the training data
for w in words[:5]:
    print(f'\n{w}')
    context = [0] * block_size # context/input initialized as list of block_size many zeros (0 is the index for the special character '.')
    for ch in w + '.':    # iterate over characters of name; name is made to end with our special character
        ix = stoi[ch]     # map character to its index
        X.append(context) # append current context to list of inputs
        Y.append(ix)      # append character's index to list of labels
        # Showing what input and expected output now look like
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        # crop and append, like a sliding window; NEAT!
        # new context starts at index 1 of the old one and is appended with the single new index,
        # coincidentally, this is the label of the input-label pair produced earlier -> sliding window over block_size many characters
        context = context[1:] + [ix] # New context is 

# again, these *do not* carry characters, but their respective indexes
X = torch.tensor(X) # block_size many character indices form the input; shape: (number of training examples, block_size)
Y = torch.tensor(Y) # the index of the next character forms the label; shape: (number of training examples,)  

print(f'Input:  {X.shape}\tdtype: {X.dtype}\tInitial example: {X[0]}') # 32 times the 3 character indices forming one input example
print(f'Output: {Y.shape}\tdtype: {Y.dtype}\tInitial example: {Y[0]}') # 32 times the index of the character after the 3 character indices of the input
        


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .

olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .

ava
... ---> a
..a ---> v
.av ---> a
ava ---> .

isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .

sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .
Input:  torch.Size([32, 3])	dtype: torch.int64	Initial example: tensor([0, 0, 0])
Output: torch.Size([32])	dtype: torch.int64	Initial example: 5


In [54]:
#begin with a lookup table C that maps each of the possible character indices  to a respective 2-dimensional vector each
#The embedding size of is chosen so we can nicely visualize the learned representations later
# This will be our lookup table, randomly initialized
C = torch.randn((27, 2)) # each of the 27 character indices gets its own unique 2-dimensional numeric embedding
#print(C)
xc = 'e'
i_xc = stoi[xc]
print(f"Embedding of character '{xc}' through its index {i_xc} in C: {C[i_xc]}")

print(F.one_hot(torch.tensor(stoi['e']), num_classes=27).float() @ C) # stoi['e'] = 5
#we won't use it. just demonstrating that the one-hot encoding of the character index, 
#when multiplied with the embedding matrix C, gives us the same embedding as directly indexing into C with the character index.
#While this is an established technique, a direct call like C[5] is sufficient for our needs.
print(F.one_hot(torch.tensor(stoi['e']), num_classes=27).float() @ C == C[stoi['e']]) 

# We can write this:
print(X[13, 2])   # Plugging out an example character, here 'a' from ..a ---> v (14th input, 3rd character)
# As the above holds, we can also write this:
print(C[X][13,2]) # This construct actually returns the 2-dim. embedding vector for 'a'

# Given the slicing technique from above, which works with entire tensors for indexing, we can therefore write this:
emb = C[X] # For each index in each input conext, we get the corresponding embedding vector from C
print(C[X].shape) # 32 times 3 times 2 -> 32 inputs, 3 characters each, 2 dimensional embedding vector each


Embedding of character 'e' through its index 5 in C: tensor([-0.0135,  0.6991])
tensor([-0.0135,  0.6991])
tensor([True, True])
tensor(1)
tensor([-2.1794, -1.3379])
torch.Size([32, 3, 2])


In [47]:
emb = C[X] # replace each character index in X with its corresponding embedding from C; shape: (number of training examples, block_size, embedding size)
print(f'Embedding of input X through C: {emb.shape}\nExample embedding for first input: {emb[0]}')

Embedding of input X through C: torch.Size([32, 3, 2])
Example embedding for first input: tensor([[ 0.4263, -0.0786],
        [ 0.4263, -0.0786],
        [ 0.4263, -0.0786]])


# We receive three character embedding vectors from embedding our training input through C.
# Each embedding vector contains two numerical values.
# Therefore, we need to consider a total of inputs per character context, provided in the form of a matrix of shape 32, 3, 2 we'll call emb
# (32 contexts in X, 3 characters per context, 2 values per character embedding).

# Below, we begin to construct a hidden layer that, per context, takes in the 3,2 shaped inputs and produces a respective output of shape 3, 100.

In [56]:
W1 = torch.randn((6, 100)) # 6 -> 3 vectors á 2 values, 100 neurons
b1 = torch.randn(100)      # for each of the 100 neurons, add a bias

In [ ]:
# h is the activated ouput of the hidden layer
# h = torch.tanh(emb.view(emb.shape[0],6) @ W1 + b1) # emb.shape[0] is 32, but we can and should make this flexible

# Flatten 3x2 to 6, so we can multiply with W1, which expects 6 inputs per neuron:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
# PyTorch reads the -1 and infers, because 6 is already used up from (2x3): 32 times 6 or inputs times 6 generally
#Ultimately, when adding the bias term like so emb.view(-1, 6) @ W1 + b1,
#all of the sets of neurons get incremented each by the same set of biases now.
#That's good, because we want to actually increment each input by the always same bias values

# Let's see what h and its contents look like
print(h.shape)  # 32 times 100 activations of the hidden layer
print(h[0][:5]) # first five activations

torch.Size([32, 100])
tensor([-0.9494, -0.3434, -0.9825,  0.3760,  0.9904])


In [62]:
# Output Layer
# We use softmax instead of tanh to obtain a normalized distribution over the 27 possible output characters
# Initialize weights and biases for the output layer randomly
W2 = torch.randn((100, 27)) # 100 inputs, 27 output neurons, 27 is fixed due to the 27 possible characters in the dataset
b2 = torch.randn(27)        # 27 biases

# Raw neuron outputs
logits = h @ W2 + b2
print('Logits:', logits.shape) # (32, 27) -> 32 outputs, 27 dimensions each

# Softmax transforms outputs into a normalized distribution over the 27 possible characters
pseudo_counts = logits.exp() # Refer to N002 for why I call this pseudo_counts and not counts
prob = pseudo_counts / pseudo_counts.sum(1, keepdims=True)
print('Probabilities:', prob.shape) # (32, 27) -> 32 distributions across 27 next character candidates each

Logits: torch.Size([32, 27])
Probabilities: torch.Size([32, 27])


In [ ]:
#Loss function
#We now have to compare the output from each of the distributions from the last layer to the respective labels found in Y
# For each of the 32 input triplets:
# Give the probability assigned to the correct character, which is accessible at the index stored in Y
#[
#    prob[0, Y[0]],
#    prob[1, Y[1]],
#    prob[2, Y[2]],
#    ...
#    prob[31, Y[31]]
#]
print(prob[torch.arange(32), Y])
print(prob[torch.arange(32), Y].shape) # 32 probabilities, one per each ground truth next character

# We want the average log proability over all 32 inputs to be our loss
loss = -prob[torch.arange(32), Y].log().mean()
print(loss.item()) # This is to be minimized


tensor([1.8211e-12, 9.9956e-01, 9.9462e-01, 1.1885e-14, 4.9905e-14, 1.6012e-16,
        1.0489e-03, 4.4989e-04, 1.6949e-07, 6.3549e-07, 8.8166e-01, 2.9971e-20,
        1.7428e-06, 8.6281e-09, 9.6321e-10, 5.1059e-12, 4.9176e-14, 4.3388e-11,
        2.1505e-04, 1.1924e-15, 7.6944e-15, 6.9569e-06, 7.3932e-03, 9.8242e-01,
        3.8972e-14, 3.6477e-12, 2.9633e-12, 2.7201e-04, 2.2778e-07, 5.5950e-06,
        1.7738e-07, 3.4565e-19])
torch.Size([32])
19.31924819946289


In [93]:
#Condensing the concepts - MLP architecture
print('X:', X.shape)
print('Y:', Y.shape, '\n')

block_size = 3 # this is the context length (how many characters are used to predict the next one)
embedding_size = 2 # this is the dimensionality of the character embeddings
hidden_layer_size = 100 # this is the number of neurons in the hidden layer

# Let's re-initialize the MLP's weights and biases
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27,embedding_size), generator=g)          #  27 characters, 2 embedding dimensions each

W1 = torch.randn((block_size * C.shape[1], hidden_layer_size), generator=g) # block_size times embedding dimension as input (3x2=6) to 100 neurons
b1 = torch.randn((hidden_layer_size), generator=g)          # 100 biases to be added to the 100 neuron outputs

W2 = torch.randn((hidden_layer_size,27), generator=g)       # 100 neuron outputs as inputs to 27 output neurons
b2 = torch.randn((27), generator=g)           #  27 biases to be added to the  27 output neurons

parameters = [C, W1, b1, W2, b2] # Group all layers' parameters into one structure
print(sum(p.nelement() for p in parameters), 'parameters') # print total #parameters (nelements = number of elements)

# Go over parameters, have them allow gradient accumulation
for p in parameters:
    p.requires_grad = True

# Run for 1000 training epochs
for _ in range(1000):
    ## Forward-Pass
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)

    # counts = logits.exp()
    # prob = counts / counts.sum(1, keepdims=True)
    # loss = -prob[torch.arange(32), Y].log().mean()
    
    # Replacing the above lines with PyTorch's built-in cross entropy loss function:
    # this function behaves better numerically and avoids pitfalls like infinities by subtracting the maximum value from all logits before exponentiating
    loss = F.cross_entropy(logits, Y)

    ## Backward-Pass
    # Clear gradients before each backward pass, avoiding accumulation across multiple passes
    for p in parameters:
        p.grad = None
    
    loss.backward()
    
    # Update parameters using gradient descent and a learning rate of 0.1
    for p in parameters:
        p.data += -0.1 * p.grad # Nudge parameter values in negative gradient direction

print(loss.item())    

X: torch.Size([32, 3])
Y: torch.Size([32]) 

3481 parameters
0.2561509907245636


In [88]:
#Comparing the logits (the actual network outputs) to the labels Y, we see that the predictions are visibly mimicking the labels
# For each of the 32 inputs, give the index of the highest probability output neuron
print(logits.max(1)) # This is what the network thinks is the respectively most likely next character
print(f'\n{Y}')

torch.return_types.max(
values=tensor([13.3437, 17.7879, 20.5832, 20.6042, 16.7390, 13.3437, 15.9747, 14.1889,
        15.9158, 18.3894, 15.9409, 20.9284, 13.3437, 17.1212, 17.1498, 20.0637,
        13.3437, 16.4564, 15.1328, 17.0537, 18.5905, 15.9655, 10.8739, 10.6874,
        15.5062, 13.3437, 16.2394, 16.9563, 12.7426, 16.2141, 19.0840, 16.0213],
       grad_fn=<MaxBackward0>),
indices=tensor([ 9, 13, 13,  1,  0,  9, 12,  9, 22,  9,  1,  0,  9, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0,  9, 15, 16,  8,  9,  1,  0]))

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])
